# 🧠 04 — LLM: Explicaciones en Lenguaje Natural
**Proyecto:** Detección de Fraude Financiero con ML + LLM Explicativo  
**Responsable:** Nombre Completo 3 · correo3@eafit.edu.co

---
**Objetivo:** Usar SHAP + un LLM (Groq/llama-3.1) para generar explicaciones en lenguaje natural de por qué una transacción fue clasificada como fraude. Estrategia: **few-shot prompting**.

**Modelo LLM usado:** `llama-3.1-8b-instant` vía Groq API (gratuito)  
**Obtener API key gratuita:** https://console.groq.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, json, joblib
warnings.filterwarnings('ignore')
from dotenv import load_dotenv

load_dotenv('../.env')  # Carga GROQ_API_KEY desde .env

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
LLM_MODEL    = os.getenv('LLM_MODEL', 'llama-3.1-8b-instant')

if GROQ_API_KEY:
    print(f'✅ GROQ_API_KEY cargada ({GROQ_API_KEY[:8]}...)')
else:
    print('⚠️  GROQ_API_KEY no encontrada en .env')
    print('   El notebook funcionará en MODO DEMO (respuestas simuladas)')
    print('   Para API real: crear .env con GROQ_API_KEY=gsk_...')

print(f'Modelo LLM: {LLM_MODEL}')

## 1. Cargar modelo y datos

In [ ]:
PROC = '../data/processed/'
MODELS = '../models/'

# Cargar modelo
if os.path.exists(MODELS + 'modelo_final.joblib'):
    modelo = joblib.load(MODELS + 'modelo_final.joblib')
    print('✅ Modelo cargado: models/modelo_final.joblib')
else:
    from xgboost import XGBClassifier
    from sklearn.datasets import make_classification
    from sklearn.preprocessing import StandardScaler
    X_s, y_s = make_classification(n_samples=5000, n_features=30, weights=[0.997, 0.003], random_state=42)
    sc = StandardScaler()
    X_s = sc.fit_transform(X_s)
    modelo = XGBClassifier(n_estimators=100, random_state=42, verbosity=0, scale_pos_weight=332)
    modelo.fit(X_s, y_s)
    print('⚠️  Modelo sintético generado. Ejecuta 03_modeling.ipynb primero.')

# Cargar test set
if os.path.exists(PROC + 'X_test.npy'):
    X_test = np.load(PROC + 'X_test.npy')
    y_test = np.load(PROC + 'y_test.npy')
    feat_csv = PROC + 'feature_names.csv'
    feat_names = pd.read_csv(feat_csv).iloc[:, 0].tolist() if os.path.exists(feat_csv) else [f'feature_{i}' for i in range(X_test.shape[1])]
    print(f'✅ Test set cargado: {X_test.shape}')
else:
    print('⚠️  Test set no encontrado. Ejecuta 02_preprocessing.ipynb primero.')
    X_test = np.random.randn(200, 30)
    y_test = np.random.choice([0,1], 200, p=[0.997, 0.003])
    feat_names = [f'feature_{i}' for i in range(30)]

## 2. Calcular SHAP values para el test set

In [ ]:
try:
    import shap
    explainer  = shap.TreeExplainer(modelo)
    n_sample   = min(200, X_test.shape[0])
    shap_vals  = explainer.shap_values(X_test[:n_sample])
    base_value = explainer.expected_value
    if isinstance(base_value, list):
        base_value = base_value[1]
    print(f'✅ SHAP calculado para {n_sample} muestras. Base value: {base_value:.4f}')
    SHAP_OK = True
except Exception as e:
    print(f'⚠️  SHAP no disponible: {e}')
    SHAP_OK = False
    n_sample = min(200, X_test.shape[0])
    shap_vals = np.random.randn(n_sample, X_test.shape[1]) * 0.1
    base_value = 0.0035

## 3. Función de llamada al LLM (Groq)

In [ ]:
def llamar_llm(prompt: str, sistema: str = None, max_tokens: int = 400) -> str:
    """
    Llama al LLM via Groq API o retorna respuesta demo si no hay API key.
    """
    if not GROQ_API_KEY:
        return (
            "[MODO DEMO — Sin API Key]\n"
            "Esta transacción fue clasificada como posible fraude principalmente porque "
            "el patrón de sus componentes V4 y V11 es inusual comparado con transacciones "
            "legítimas: V4 muestra un valor anormalmente bajo (-3.2) mientras que V11 es "
            "elevado (+2.8), una combinación presente en el 94% de los fraudes del conjunto "
            "de entrenamiento. El monto (log_amount=5.1, equivalente a ~$164) es moderado, "
            "pero la combinación temporal (hora 2am) y el perfil de componentes PCA "
            "sugieren alta sospecha. Recomendación: bloquear y notificar al titular."
        )
    try:
        from groq import Groq
        client   = Groq(api_key=GROQ_API_KEY)
        messages = []
        if sistema:
            messages.append({'role': 'system', 'content': sistema})
        messages.append({'role': 'user', 'content': prompt})

        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f'[Error al llamar al LLM: {e}]'

print('✅ Función llamar_llm lista')

## 4. Sistema de prompting — Few-shot

In [ ]:
SISTEMA = """Eres un analista experto en detección de fraude financiero.
Tu tarea es explicar en lenguaje claro y conciso (máximo 120 palabras) por qué
un modelo de machine learning clasificó una transacción como sospechosa.
Basa tu explicación ÚNICAMENTE en los valores SHAP proporcionados.
Usa un tono profesional pero accesible. No inventes datos adicionales.
Termina con una recomendación de acción (bloquear / revisar / aprobar)."""

FEW_SHOT_EXAMPLES = """
=== EJEMPLO 1 (fraude confirmado) ===
Features más relevantes:
- V4: -3.21 → contribuye +0.42 a fraude (valor muy bajo es señal de fraude)
- V11: +2.87 → contribuye +0.31 a fraude
- log_amount: 5.10 ($164) → contribuye -0.05 (monto moderado, neutral)
- hour_of_day: 2.3h → contribuye +0.18 a fraude (madrugada)
Probabilidad de fraude: 0.94

Explicación esperada:
Esta transacción presenta señales claras de fraude. El componente V4 tiene un valor
inusualmente bajo (-3.21), patrón asociado con el 89% de los fraudes históricos.
Combinado con V11 elevado y el horario de madrugada (2:18am), el modelo asigna
una probabilidad de fraude del 94%. El monto de $164 no es determinante.
Recomendación: BLOQUEAR y notificar al titular de forma inmediata.

=== EJEMPLO 2 (transacción legítima) ===
Features más relevantes:
- V4: +1.12 → contribuye -0.15 (valor normal, reduce sospecha)
- V14: -0.43 → contribuye -0.08
- log_amount: 3.22 ($25) → contribuye -0.12 (monto bajo, típico)
- hour_of_day: 14.5h → contribuye -0.09 (horario normal)
Probabilidad de fraude: 0.03

Explicación esperada:
Esta transacción muestra características consistentes con compras legítimas.
Los componentes PCA están dentro de rangos normales y el monto de $25 a las 2:30pm
es un patrón frecuente en el historial legítimo. La probabilidad de fraude es 3%.
Recomendación: APROBAR sin intervención adicional.
"""

print('✅ Sistema de prompting few-shot definido')

## 5. Función de explicación completa

In [ ]:
def explicar_transaccion(idx: int, X: np.ndarray, shap_values: np.ndarray,
                          feature_names: list, modelo, top_k: int = 6) -> dict:
    """
    Genera una explicación en lenguaje natural para la transacción en índice idx.

    Retorna dict con: idx, prediccion, probabilidad, shap_top, explicacion_llm
    """
    valores_x   = X[idx]
    shap_fila   = shap_values[idx] if len(shap_values.shape) == 2 else shap_values[idx, :, 1]
    prob_fraude = float(modelo.predict_proba(X[idx:idx+1])[0, 1])
    pred        = int(prob_fraude >= 0.5)

    # Top-k features por impacto SHAP absoluto
    top_idx  = np.argsort(np.abs(shap_fila))[::-1][:top_k]
    shap_top = [
        {
            'feature':     feature_names[i] if i < len(feature_names) else f'f{i}',
            'valor':       round(float(valores_x[i]), 4),
            'shap':        round(float(shap_fila[i]), 4),
            'direccion':   'aumenta sospecha' if shap_fila[i] > 0 else 'reduce sospecha'
        }
        for i in top_idx
    ]

    # Construir prompt
    features_str = '\n'.join(
        f"- {r['feature']}: {r['valor']:+.3f} → SHAP {r['shap']:+.4f} ({r['direccion']})"
        for r in shap_top
    )
    prompt = f"""{FEW_SHOT_EXAMPLES}
=== TRANSACCIÓN A EXPLICAR ===
Features más relevantes (SHAP):
{features_str}
Probabilidad de fraude: {prob_fraude:.2f}
Predicción del modelo: {'FRAUDE' if pred == 1 else 'LEGÍTIMA'}

Genera la explicación:"""

    explicacion = llamar_llm(prompt, sistema=SISTEMA)

    return {
        'idx':           idx,
        'prediccion':    'FRAUDE' if pred == 1 else 'LEGÍTIMA',
        'probabilidad':  round(prob_fraude, 4),
        'shap_top':      shap_top,
        'explicacion':   explicacion
    }

print('✅ Función explicar_transaccion lista')

## 6. Demostración — 3 casos explicados

In [ ]:
# Seleccionar casos interesantes
y_prob_all = modelo.predict_proba(X_test[:n_sample])[:, 1]

idx_alta_prob  = int(np.argmax(y_prob_all))                          # Máxima probabilidad de fraude
idx_baja_prob  = int(np.argmin(y_prob_all))                          # Mínima probabilidad
idx_borderline = int(np.argmin(np.abs(y_prob_all - 0.5)))           # Caso borderline

casos = [
    (idx_alta_prob,  '🔴 Caso 1: Alta probabilidad de fraude'),
    (idx_baja_prob,  '🟢 Caso 2: Baja probabilidad (legítima)'),
    (idx_borderline, '🟡 Caso 3: Caso borderline (límite de decisión)'),
]

explicaciones = []
for idx, titulo in casos:
    print(f'\n{'='*60}')
    print(f'{titulo}')
    print('='*60)
    resultado = explicar_transaccion(idx, X_test, shap_vals, feat_names, modelo)
    explicaciones.append(resultado)

    print(f"Predicción: {resultado['prediccion']}  |  Probabilidad: {resultado['probabilidad']:.4f}")
    print(f"\nTop features SHAP:")
    for r in resultado['shap_top']:
        print(f"  {r['feature']:15s} val={r['valor']:+.3f}  SHAP={r['shap']:+.4f}  → {r['direccion']}")
    print(f"\n💬 Explicación LLM:")
    print(resultado['explicacion'])

## 7. Guardar explicaciones

In [ ]:
# Serializar (quitar arrays numpy para JSON)
def serializable(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

explic_json = json.loads(json.dumps(explicaciones, default=serializable))

os.makedirs('../models', exist_ok=True)
with open('../models/explicaciones_llm.json', 'w', encoding='utf-8') as f:
    json.dump(explic_json, f, ensure_ascii=False, indent=2)

print('✅ Explicaciones guardadas: models/explicaciones_llm.json')
print('\n→ El sistema completo está listo.')
print('→ Para la demo interactiva: streamlit run app/app.py')